# 🛡️ Ranking de Cidades Inteligentes: Foco em Segurança
Objetivo: Processar indicadores urbanos do dataset para criar um score de segurança.

Metodologia: Pipeline de ETL focado em limpeza de strings, imputação estatística (mediana) e normalização Min-Max com pesos focados em infraestrutura e políticas públicas.

In [5]:
import pandas as pd
import numpy as np

In [10]:
# ETAPA 1: INGESTÃO DE DADOS
# Leitura direta ignorando as primeiras linhas de metadados do arquivo.
# Foco em eficiência, carregando apenas o bloco tabular útil.
# =====================================================================
print("Carregando os dados brutos...")
df = pd.read_csv("Cidade Inteligente(DESSUST).csv", sep=';', skiprows=2, encoding='utf-8')

Carregando os dados brutos...


In [11]:
# ETAPA 2: HIGIENIZAÇÃO DE CABEÇALHOS
# Resolve o problema crônico de dados governamentais: quebras de linha 
# (\n) e espaços invisíveis ocultos nos nomes das colunas.
# =====================================================================
print("Higienizando esquema de metadados...")
df.columns = df.columns.str.replace('\n', ' ').str.replace('\r', '').str.strip()

colunas_projeto = [
    'Cidade', 'Estado', 'População total estimada do município',
    'Taxa de homicídios', 
    'Soluções em monitoramento para a segurança pública', 
    'Políticas públicas e ações para segurança pública',
    'Urbanização vias públicas', 
    'Soluções para telegestão da iluminação pública',
    'Taxa de analfabetismo', 
    'Geração de trabalho e renda no município', 
    'Índice de desenvolvimento humano do município (IDH-M)',
    'Pesquisas de satisfação/ Monitoramento da percepção de qualidade'
]

Higienizando esquema de metadados...


In [12]:
# ETAPA 3: FILTRAGEM SEGURA E SELEÇÃO
# Utilizando filter() para evitar a quebra do pipeline (KeyError) 
# caso a fonte de dados mude ou perca colunas no futuro.
# =====================================================================
print("Extraindo features de interesse...")
df_trabalho = df.filter(items=colunas_projeto).copy()

Extraindo features de interesse...


In [13]:
# ETAPA 4: LIMPEZA DOS DADOS E TIPAGEM
# Higienização de strings, conversão decimal e coerção de tipos.
# =====================================================================
print(" Realizando limpeza de nulos e coerção de tipagem...")

# Substituindo strings categóricas irrelevantes ou de ausência por NaN
df_trabalho = df_trabalho.replace(['ND', 'NÃO', 'NAO', 'SIM', ''], np.nan)

# Isolando apenas as colunas numéricas para a transformação
colunas_numericas = df_trabalho.columns.drop(['Cidade', 'Estado'])

for col in colunas_numericas:
    # Padronizando separador decimal do formato BR (,) para US (.)
    df_trabalho[col] = df_trabalho[col].astype(str).str.replace(',', '.')
    
    # Forçando tipagem numérica (valores corrompidos viram NaN de forma segura)
    df_trabalho[col] = pd.to_numeric(df_trabalho[col], errors='coerce')

# Imputação de dados: optamos pela Mediana por ser estatisticamente 
# mais robusta a outliers (comum devido à diferença populacional das cidades).
df_trabalho[colunas_numericas] = df_trabalho[colunas_numericas].fillna(df_trabalho[colunas_numericas].median())

 Realizando limpeza de nulos e coerção de tipagem...


In [15]:
# ETAPA 5: ENGENHARIA DE FEATURES (CRIANDO O RANKING)
# Normalização Min-Max implementada nativamente (sem dependência do 
# scikit-learn) para reduzir o footprint de memória e processamento.
# =====================================================================
print("Processando a matemática do Score de Segurança...")

df_norm = pd.DataFrame()

for col in colunas_numericas:
    min_val = df_trabalho[col].min()
    max_val = df_trabalho[col].max()
    
    # Prevenção de erro de divisão por zero em colunas constantes
    if max_val - min_val == 0:
        df_norm[col] = 0
    else:
        df_norm[col] = (df_trabalho[col] - min_val) / (max_val - min_val)

# ATENÇÃO: Inversão de Polaridade
# Taxa de homicídios e Analfabetismo afetam negativamente a segurança.
# A métrica (1 - x) garante que "menores taxas" gerem "maiores notas".
df_norm['Taxa de homicídios'] = 1 - df_norm['Taxa de homicídios']
df_norm['Taxa de analfabetismo'] = 1 - df_norm['Taxa de analfabetismo']

# PESOS DO RANKING (Total = 1.0)
pesos = {
    'Taxa de homicídios': 0.35,
    'Soluções em monitoramento para a segurança pública': 0.10,
    'Políticas públicas e ações para segurança pública': 0.10,
    'Urbanização vias públicas': 0.10,
    'Soluções para telegestão da iluminação pública': 0.10,
    'Taxa de analfabetismo': 0.05,
    'Geração de trabalho e renda no município': 0.05,
    'Índice de desenvolvimento humano do município (IDH-M)': 0.10,
    'Pesquisas de satisfação/ Monitoramento da percepção de qualidade': 0.05
}

# Calculando a pontuação final de 0 a 100
df_trabalho['Score_Seguranca'] = 0
for col, peso in pesos.items():
    if col in df_norm.columns:
        df_trabalho['Score_Seguranca'] += (df_norm[col] * peso) * 100

Processando a matemática do Score de Segurança...


In [ ]:
# ETAPA 6: ORDENAÇÃO E EXPORTAÇÃO DO ARTEFATO
# Filtro de relevância populacional e dump dos dados para consumo.
# =====================================================================
print("📦 Empacotando e exportando o dado refinado...")

# Ordenar do maior para o menor Score
df_final = df_trabalho.sort_values(by='Score_Seguranca', ascending=False).reset_index(drop=True)

# Corte metodológico: cidades com mais de 50 mil habitantes
df_final = df_final[df_final['População total estimada do município'] >= 50000]

print("\n🏆 Amostra das Cidades Top 5:")
print(df_final[['Cidade', 'Estado', 'Score_Seguranca']].head())

# Salvar o artefato final para ser consumido pelo Dashboard
df_final.to_csv('ranking_seguranca_cidades.csv', index=False, encoding='utf-8')
print("\n✅ Pipeline concluído. Artefato salvo com sucesso!")

📦 Empacotando e exportando o dado refinado...

🏆 Amostra do Top 5 Cidades:
            Cidade Estado  Score_Seguranca
0  Poços de Caldas     MG        77.863836
1         Botucatu     SP        77.805679
2          Maringá     PR        77.661708
3         Sorocaba     SP        77.290776
4         Blumenau     SC        76.926000

✅ Pipeline concluído. Artefato salvo com sucesso!


In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# ETAPA 1: INGESTÃO DE DADOS
# Utilizamos skiprows=2 para pular as linhas de metadados do topo do CSV
# ---------------------------------------------------------
print("📥 Carregando os dados...")
df = pd.read_csv("Cidade Inteligente(DESSUST).csv", sep=';', skiprows=2, encoding='utf-8')

# ---------------------------------------------------------
# ETAPA 2: HIGIENIZAÇÃO DE CABEÇALHOS (A Solução da Etapa 5)
# Removemos quebras de linha (\n) e espaços extras que quebram o código
# ---------------------------------------------------------
df.columns = df.columns.str.replace('\n', ' ').str.replace('\r', '').str.strip()

colunas_projeto = [
    'Cidade', 'Estado', 'População total estimada do município',
    'Taxa de homicídios', 
    'Soluções em monitoramento para a segurança pública', 
    'Políticas públicas e ações para segurança pública',
    'Urbanização vias públicas', 
    'Soluções para telegestão da iluminação pública',
    'Taxa de analfabetismo', 
    'Geração de trabalho e renda no município', 
    'Índice de desenvolvimento humano do município (IDH-M)',
    'Pesquisas de satisfação/ Monitoramento da percepção de qualidade' # Agora vai funcionar!
]

# ---------------------------------------------------------
# ETAPA 3: FILTRAGEM SEGURA E SELEÇÃO
# filter() evita que o script quebre caso uma coluna não exista
# ---------------------------------------------------------
print("🗂️ Filtrando colunas...")
df_trabalho = df.filter(items=colunas_projeto).copy()

# ---------------------------------------------------------
# ETAPA 4: LIMPEZA DOS DADOS E TIPAGEM
# Substituir lixo de string, trocar vírgula por ponto e converter
# ---------------------------------------------------------
print("🧹 Limpando dados (ND, Nulos e Tipagem)...")
# Remover 'ND' e afins
df_trabalho = df_trabalho.replace(['ND', 'NÃO', 'NAO', 'SIM', ''], np.nan)

# Separar numericas para tratamento (excluindo Cidade e Estado)
colunas_numericas = df_trabalho.columns.drop(['Cidade', 'Estado'])

for col in colunas_numericas:
    # Trocar vírgula por ponto nas strings numéricas brasileiras
    df_trabalho[col] = df_trabalho[col].astype(str).str.replace(',', '.')
    # Converter para numérico (o que der erro vira NaN)
    df_trabalho[col] = pd.to_numeric(df_trabalho[col], errors='coerce')

# Imputação de dados faltantes usando a Mediana (mais robusto a outliers)
df_trabalho[colunas_numericas] = df_trabalho[colunas_numericas].fillna(df_trabalho[colunas_numericas].median())

# ---------------------------------------------------------
# ETAPA 5: ENGENHARIA DE FEATURES (Criando o Ranking)
# Normalizamos os dados de 0 a 1 para poder somar bananas com laranjas
# ---------------------------------------------------------
print("⚙️ Calculando o Score de Segurança...")

# Normalização Min-Max manual para não precisar importar o Scikit-Learn (economia de recursos)
df_norm = pd.DataFrame()

for col in colunas_numericas:
    min_val = df_trabalho[col].min()
    max_val = df_trabalho[col].max()
    
    # Prevenção de divisão por zero
    if max_val - min_val == 0:
        df_norm[col] = 0
    else:
        df_norm[col] = (df_trabalho[col] - min_val) / (max_val - min_val)

# ATENÇÃO: Inversão de Polaridade
# Taxa de homicídios e Analfabetismo são coisas ruins. Menos é melhor.
# Subtraímos de 1 para que "menos crime" = "nota maior".
df_norm['Taxa de homicídios'] = 1 - df_norm['Taxa de homicídios']
df_norm['Taxa de analfabetismo'] = 1 - df_norm['Taxa de analfabetismo']

# PESOS DO RANKING (Total = 1.0)
# Você pode ajustar esses pesos conforme a regra de negócio
pesos = {
    'Taxa de homicídios': 0.35,
    'Soluções em monitoramento para a segurança pública': 0.10,
    'Políticas públicas e ações para segurança pública': 0.10,
    'Urbanização vias públicas': 0.10,
    'Soluções para telegestão da iluminação pública': 0.10,
    'Taxa de analfabetismo': 0.05,
    'Geração de trabalho e renda no município': 0.05,
    'Índice de desenvolvimento humano do município (IDH-M)': 0.10,
    'Pesquisas de satisfação/ Monitoramento da percepção de qualidade': 0.05
}

# Calculando a pontuação final (Score de 0 a 100)
df_trabalho['Score_Seguranca'] = 0
for col, peso in pesos.items():
    if col in df_norm.columns:
        df_trabalho['Score_Seguranca'] += (df_norm[col] * peso) * 100

# ---------------------------------------------------------
# ETAPA 6: ORDENAÇÃO E EXPORTAÇÃO
# ---------------------------------------------------------
# Ranquear do maior Score para o menor
df_final = df_trabalho.sort_values(by='Score_Seguranca', ascending=False).reset_index(drop=True)

# Opcional: Filtrar apenas cidades com mais de 50 mil habitantes para um ranking mais justo
df_final = df_final[df_final['População total estimada do município'] >= 50000]

print("🏆 Top 5 Cidades:")
print(df_final[['Cidade', 'Estado', 'Score_Seguranca']].head())

# Salvar
df_final.to_csv('ranking_seguranca_cidades.csv', index=False, encoding='utf-8')
print("✅ Arquivo 'ranking_seguranca_cidades.csv' salvo com sucesso!")